In [1]:
from dataset_greater_than import YearDataset
from transformers import AutoTokenizer
from utils_greater_than import get_valid_years
from pathlib import Path
import json


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/workspace/feature-circuits/venv/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/workspace/feature-circuits/venv/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/workspace/feature-circuits/venv/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_lo

In [3]:
import json

input_path = Path("data/greater_than_examples.json")
output_path = Path("data/greater_than_examples_fake.json")
processed_data = []

with input_path.open("r") as f:
    for line in f:
        example = json.loads(line)
        example["clean_answer"] = "ab"
        example["patch_answer"] = "cd"
        processed_data.append(example)

with output_path.open("w") as f:
    for example in processed_data:
        json.dump(example, f)
        f.write("\n")



In [2]:
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-70m-deduped")
years_to_sample_from = get_valid_years(tokenizer, 1000, 1900)

/workspace/feature-circuits/venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [3]:
N = 10000  
ds = YearDataset(
    years_to_sample_from,
    N,
    Path("cache/potential_nouns.txt"),
    tokenizer,
    balanced=False,
    eos=False,
    device='cuda:0',
)

/workspace/feature-circuits/dataset_greater_than.py:102: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.years = torch.tensor(self.years_to_sample_from[torch.randint(0, len(self.years_to_sample_from), (N,))])


In [4]:
ds.good_sentences

['The chaos lasted from the year 1294 to the year 12',
 'The exchange lasted from the year 1473 to the year 14',
 'The voyage lasted from the year 1524 to the year 15',
 'The assaults lasted from the year 1836 to the year 18',
 'The existence lasted from the year 1451 to the year 14',
 'The modernization lasted from the year 1072 to the year 10',
 'The disagreement lasted from the year 1436 to the year 14',
 'The cooperation lasted from the year 1650 to the year 16',
 'The plague lasted from the year 1689 to the year 16',
 'The conspiracy lasted from the year 1627 to the year 16',
 'The convention lasted from the year 1413 to the year 14',
 'The assaults lasted from the year 1705 to the year 17',
 'The engagement lasted from the year 1231 to the year 12',
 'The tour lasted from the year 1665 to the year 16',
 'The project lasted from the year 1834 to the year 18',
 'The endeavor lasted from the year 1232 to the year 12',
 'The disagreement lasted from the year 1221 to the year 12',
 'T

In [5]:
ds.years_YY

tensor([94, 73, 24,  ..., 79, 77, 46])

In [6]:
ds.years_XX

tensor([12, 14, 15,  ..., 17, 14, 14])

In [7]:
str(next(y for y in range(14 * 100 + 24 + 1, 14 * 100 + 99) if y in years_to_sample_from))

'1425'

In [8]:
output_path = Path("data/greater_than_examples.json")
with output_path.open("w") as f:
    for clean_prefix, patch_prefix, year_prefix, year_suffix in zip(ds.good_sentences, ds.bad_sentences, ds.years_XX, ds.years_YY):
        try:
            clean_answer = str(next(y for y in range(year_prefix * 100 + year_suffix + 1, year_prefix * 100 + 99) if y in years_to_sample_from))[2:]
            data = {
                "clean_prefix": clean_prefix,
                "patch_prefix": patch_prefix,
                "clean_answer": clean_answer,
                "patch_answer": "01",
                "case": "greater_than_dataset"
            }
            clean_prefix = clean_prefix + clean_answer
            patch_prefix = patch_prefix + "01"
            if len(tokenizer.encode(clean_prefix)) == 13 and len(tokenizer.encode(patch_prefix)) == 13:
                f.write(json.dumps(data) + "\n")
        except StopIteration:
            print(f"No year found for {year_prefix}{year_suffix}")

print(f"Data saved to {output_path}")


No year found for 1298
No year found for 1498
No year found for 1112
No year found for 1098
No year found for 1798
No year found for 1380
No year found for 1798
No year found for 1698
No year found for 1380
No year found for 1380
No year found for 1879
No year found for 1598
No year found for 1879
No year found for 1380
No year found for 1098
No year found for 1598
No year found for 1498
No year found for 1298
No year found for 1698
No year found for 1498
No year found for 1098
No year found for 1598
No year found for 1112
No year found for 1298
No year found for 1098
No year found for 1112
No year found for 1380
No year found for 1298
No year found for 1598
No year found for 1298
No year found for 1598
No year found for 1298
No year found for 1112
No year found for 1298
No year found for 1798
No year found for 1698
No year found for 1112
No year found for 1498
No year found for 1698
No year found for 1098
No year found for 1498
No year found for 1098
No year found for 1879
No year fou

In [5]:
ds.bad_sentences

['The reign lasted from the year 1701 to the year 17',
 'The assaults lasted from the year 1401 to the year 14',
 'The increase lasted from the year 1701 to the year 17',
 'The impact lasted from the year 1501 to the year 15',
 'The construction lasted from the year 1001 to the year 10',
 'The assaults lasted from the year 1601 to the year 16',
 'The construction lasted from the year 1001 to the year 10',
 'The abduction lasted from the year 1001 to the year 10',
 'The epidemic lasted from the year 1401 to the year 14',
 'The slump lasted from the year 1701 to the year 17',
 'The challenge lasted from the year 1301 to the year 13',
 'The competition lasted from the year 1501 to the year 15',
 'The trial lasted from the year 1701 to the year 17',
 'The progress lasted from the year 1801 to the year 18',
 'The dynasty lasted from the year 1401 to the year 14',
 'The collaboration lasted from the year 1201 to the year 12',
 'The campaign lasted from the year 1501 to the year 15',
 'The ca

In [7]:
ds.good_prompt

['The',
 'NOUN',
 'lasted',
 'from',
 'the',
 'year',
 'XX1',
 'YY',
 'to',
 'the',
 'year',
 'XX2']